In [ ]:
import gspread
import google.auth

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import sqlite3
import os

from dotenv import load_dotenv

load_dotenv()


True

In [ ]:
json_cred_filename = os.getenv("JSON_CREDENTIALS_FILENAME")

gc = gspread.service_account(filename=json_cred_filename)

# Abrindo a planilha pelo nome ou URL
planilha = gc.open("Tá Na Mesa - Pesquisa de Campo")
aba = planilha.sheet1

DefaultCredentialsError: Your default credentials were not found. To set up Application Default Credentials, see https://cloud.google.com/docs/authentication/external/set-up-adc for more information.

In [ ]:
planilha = aba.get_all_values()
df = pd.DataFrame(planilha)
df = df.rename(columns=df.iloc[0]).drop(df.index[0])

df_columns = list(df.columns)
df_columns[0] = "Submission ID"
df_columns[1] = "Respondent ID"
df_columns[2] = "Submitted at"

df.columns = df_columns

In [ ]:
# Remove colunas vazias de verdade: valores em branco, espaços ou totalmente nulos.
df.replace(r'^\s*$', np.nan, regex=True, inplace=True)
df = df.dropna(how="all", axis=1)
df = df.loc[:, df.columns.astype(str).str.strip() != ""]

df['Indice manual'] = range(1, len(df) + 1)

len(df.columns)

In [ ]:
df = df.drop(columns=[col for col in df.columns if col == ""], errors='ignore')

# Ou, se quiser remover colunas que são "Unnamed" (comum no Pandas)
df = df.loc[:, ~df.columns.str.contains('^Unnamed', na=False)]
df

In [ ]:
def get_dia(x):
    return (x - data_primeira_sub).days + 1

def get_semana(x):
    return ((x - data_primeira_sub).days // 7 ) + 1


df['Submitted at'] = pd.to_datetime(df['Submitted at'])


data_primeira_sub = df['Submitted at'].min()

# Criação da coluna Submitted_at_dia
df['Submitted at_dia'] = df['Submitted at'].apply(lambda x : get_dia(x))

# Criação da coluna Submitted_at_semana
df['Submitted at_semana'] = df['Submitted at'].apply(lambda x : get_semana(x))

In [ ]:
# Criação do KPI_1 V.A.G.
opcoes_q6 = [
    'Sim, deixei de comer por falta de dinheiro',
    'Sim, frequentemente deixo de fazer alguma refeição por falta de dinheiro'
]

condicao_q5 = df['Nos últimos meses, você teve medo da comida da sua casa acabar antes de ter dinheiro para comprar mais?'] == 'Sim, tive medo da comida acabar'
condicao_q6 = df['Nos últimos meses, você sentiu fome e não comeu por falta de dinheiro?'].isin(opcoes_q6)

df['kpi_vag'] = np.where(condicao_q5 & condicao_q6, 1, 0)

In [ ]:
# Criação do KPI_7 Dependência dos beneficiários em relação ao programa

opcoes_q10 = [
    'Faço apenas a refeição servida pelo programa', 'Faço apenas 2(duas) refeições por dia, contando com a refeição servida pelo programa'
]
condicao_q10 = df['Contando com a refeição distribuída pelo programa Tá Na Mesa, quantas refeições você faz por dia?'].isin(opcoes_q10)
condicao_q9 = df['Sem o programa Tá Na Mesa, você continua se alimentando normalmente?'] == 'Não, dependo totalmente da refeição servida pelo programa'


df['kpi_dependencia'] = np.where(condicao_q9 & condicao_q10, 1, 0)

In [ ]:
def get_peso(x):
    if x == 'De 4(quatro) à 5(cinco) vezes por semana': return 4.5
    elif x == 'De 2(duas) à 3(três) vezes por semana': return 2.5
    elif x == 'Cerca de 1(uma) vez por semana': return 1
    elif x == 'Quinzenalmente': return 0.5

# preparação do dataframe para cálculo dos KPIs 2 e 3, que, na verdade, ambos são valores numéricos

df['peso_frequencia_semanal'] = df['Com que frequência você consegue receber alguma refeição do programa Tá Na Mesa?'].apply(lambda x : get_peso(x))
df

In [ ]:
def get_peso_tempo_espera(x):
    if x == "Não preciso esperar, recebo a refeição rapidamente": return 0.25
    elif x == "Espero até 30(trinta) minutos": return 0.5
    elif x == "Espero de 30(trinta) minutos à 1(uma) hora": return 0.75
    elif x == "Espero de 1(uma) hora à 2(duas) horas": return 1.5
    elif x == "Espero por mais de 2(duas) horas": return 2.5


df['peso_tempo_espera'] = df['Quanto tempo você espera na fila para receber a sua refeição?'].apply(lambda x : get_peso_tempo_espera(x))
df['peso_tempo_espera']

In [ ]:
# preparação para cálculo dos KPIs 4 e 4.1, que, na verdade, ambos são valores numéricos

df['ja_ficou_sem_receber_refeicao_mesmo_na_fila'] = df['Você já esperou na fila e mesmo assim ficou sem receber a refeição?'].apply(lambda x :
    0 if (x == 'Nunca, sempre que entrei na fila recebi a refeição' or x == 'Nunca, até hoje, sempre que entrei na fila recebi a refeição') else 1)
df

In [ ]:
df['Total diário de submissões'] = df.groupby('Submitted at_dia')['Submitted at_dia'].transform('count')

totais_diarios = df.groupby('Submitted at_dia')['Total diário de submissões'].first()
total_acumulado = totais_diarios.cumsum()

df['Total acumulado de submissões por dia'] = df['Submitted at_dia'].map(total_acumulado)

df['Média diária de submissões'] = df['Total acumulado de submissões por dia'] // df['Submitted at_dia']
df[['Total diário de submissões', 'Total acumulado de submissões por dia', 'Submitted at_dia', 'Média diária de submissões']]

In [ ]:
df['Total semanal de submissões'] = df.groupby('Submitted at_semana')['Submitted at_semana'].transform('count')

totais_semanais= df.groupby('Submitted at_semana')['Total semanal de submissões'].first()
total_acumulado = totais_semanais.cumsum()

df['Total acumulado de submissões por semana'] = df['Submitted at_semana'].map(total_acumulado)

df['Média semanal de submissões'] = df['Total acumulado de submissões por semana'] // df['Submitted at_semana']
df[['Total semanal de submissões', 'Total acumulado de submissões por semana', 'Submitted at_semana', 'Média semanal de submissões']]

In [ ]:
df["Total de beneficiários"] = len(df)*1

In [ ]:
df.replace("", float("NaN"), inplace=True)
df.dropna(how="all", axis=1, inplace=True)

In [ ]:
df[df["Submission ID"] == "jePQQMY"]

In [ ]:
df.columns

In [ ]:
len(df.columns)

In [ ]:
import pandas as pd
import sqlite3
from sqlalchemy import create_engine
import os

string_conexao = os.getenv("RETOOL_CREDENTIALS")
engine_postgres = create_engine(string_conexao)

# 5. Enviar o DataFrame inteiro para o PostgreSQL
print("Enviando dados para o PostgreSQL no Retool...")

# O parâmetro if_exists='replace' cria a tabela do zero ou substitui a existente.
# Você também pode usar 'append' para apenas adicionar novas linhas.
df.to_sql(
    name='main', 
    con=engine_postgres, 
    if_exists='replace', 
    index=False
)

In [ ]:
#db_path = "./db/test_db.sqlite"
#conn = sqlite3.connect(db_path)

#df.to_sql('main', conn, if_exists='replace', index=False)

#conn.close()